# 🐦 Arabic Twitter Sentiment Analysis (Bidirectional LSTM)

Binary sentiment classification (positive / negative) of Arabic tweets, comparing a classical baseline against a deep learning model.

**Dataset:** [Arabic Sentiment Twitter Corpus](https://www.kaggle.com/datasets/mksaad/arabic-sentiment-twitter-corpus) (Kaggle, downloaded in-notebook via `kagglehub`) — ~45k labeled training tweets, ~11.5k test tweets.

**Approaches compared:**
1. **Baseline:** TF-IDF + Logistic Regression
2. **First deep learning attempt:** a plain (unidirectional) LSTM — failed to converge
3. **Final model:** Tokenizer + Embedding + **Bidirectional** LSTM

## Load the Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mksaad/arabic-sentiment-twitter-corpus")

print("Path to dataset files:", path)

In [ ]:
path

In [ ]:
import pandas as pd

In [ ]:
import os
os.listdir(path)

In [ ]:
pos_train = pd.read_csv(path + "/train_Arabic_tweets_positive_20190413.tsv", sep='\t', header=None)
neg_train = pd.read_csv(path + "/train_Arabic_tweets_negative_20190413.tsv", sep='\t', header=None)
test_pos = pd.read_csv(path + "/test_Arabic_tweets_positive_20190413.tsv", sep='\t', header=None)
test_neg = pd.read_csv(path + "/test_Arabic_tweets_negative_20190413.tsv", sep='\t', header=None)

train_data = pd.concat([pos_train, neg_train], ignore_index=True)
test_data = pd.concat([test_pos, test_neg], ignore_index=True)

# Shuffle
train_data = train_data.sample(frac=1, random_state=42).reset_index(drop=True)
test_data = test_data.sample(frac=1, random_state=42).reset_index(drop=True)

for df in [train_data, test_data]:
    df.columns = ['label', 'tweet']

train_data

## Text Cleaning

Strips URLs, mentions/hashtags, and Arabic diacritics; normalizes Alef/Taa Marbouta/Alef Maksoura variants; collapses repeated characters (e.g. `هههههههه` → `هه`); and removes punctuation/digits/emoji.

In [ ]:
import re

def clean_arabic_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Remove URLs (http, https, www)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # 2. Remove Mentions (@username) and Hashtags (#topic)
    text = re.sub(r"@\S+", "", text)
    text = re.sub(r"#\S+", "", text)  # Note: you can keep the hashtag word and remove only the '#' via text.replace('#', '')

    # 3. Remove Arabic diacritics (Fatha, Damma, Kasra, Sukun, Shadda, Tatweel/Kashida)
    arabic_diacritics = re.compile(r"""
                                     # Formatted spaces if any
                                     [ـ\u064B-\u0652] # Diacritics and Kashida
                                 """, re.VERBOSE)
    text = re.sub(arabic_diacritics, "", text)

    # 4. Text normalization (standardize Arabic characters)
    text = re.sub(r"[أإآ]", "ا", text)  # Normalize Alef forms to bare Alef
    text = re.sub(r"ة\b", "ه", text)    # Convert Taa Marbouta to Haa at the end of words
    text = re.sub(r"ى\b", "ي", text)    # Convert Alef Maksoura to Yaa at the end of words

    # 5. Remove character elongation / repetition (e.g. هههههههه -> هه, جمييييل -> جميل)
    # Reduces any character repeating more than twice consecutively down to two
    text = re.sub(r'(.)\1+', r'\1\1', text)

    # 6. Remove punctuation, numbers, emoji, and special characters
    text = re.sub(r"[^\w\s\d]", " ", text) # symbols/punctuation
    text = re.sub(r"[_\d]+", " ", text)      # digits and underscores

    # 7. Clean up extra whitespace left by the previous steps
    text = re.sub(r"\s+", " ", text).strip()

    return text

# ----- Quick test -----
sample_tweet = "الخدمممّة كانتتتت رآآئعة جداً وممتاااازة!!! 😍 شاهد الرابط التالي: https://example.com @Aya_user #تطوير"
print("Before Cleaning:", sample_tweet)
print("After Cleaning:", clean_arabic_text(sample_tweet))

In [ ]:
train_data['tweet'] = train_data['tweet'].apply(clean_arabic_text)
test_data['tweet'] = test_data['tweet'].apply(clean_arabic_text)
train_data['tweet']

## Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(train_data['label'])   # 'pos' -> 1, 'neg' -> 0
y_test = le.transform(test_data['label'])

## 1. Baseline: TF-IDF + Logistic Regression

A classical bag-of-words baseline (unigrams + bigrams) to compare the deep learning models against.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(train_data['tweet'])
X_test_tfidf = vectorizer.transform(test_data['tweet'])

baseline_model = LogisticRegression()
baseline_model.fit(X_train_tfidf, y_train)

baseline_preds = baseline_model.predict(X_test_tfidf)
print(classification_report(y_test, baseline_preds))

## 2. Deep Learning: Tokenizer + Embedding + Bidirectional LSTM

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenization
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_data['tweet'])

X_train = tokenizer.texts_to_sequences(train_data['tweet'])
X_test = tokenizer.texts_to_sequences(test_data['tweet'])

# Padding
max_len = 100
X_train = pad_sequences(X_train, maxlen=max_len, padding='post')
X_test = pad_sequences(X_test, maxlen=max_len, padding='post')

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.models import Sequential

model = Sequential([
    Embedding(input_dim=20000, output_dim=128),
    Bidirectional(LSTM(64, return_sequences=False)),  # reads the sequence in both directions
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

In [ ]:
def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(padded)

    if pred[0][0] > 0.5:
        return le.inverse_transform([1])[0]
    else:
        return le.inverse_transform([0])[0]

## 3. A Failed Attempt: Plain (Unidirectional) LSTM

Before switching to the bidirectional model above, a plain single-direction `LSTM(64)` was tried with the same tokenizer/embedding setup. It failed to converge: training accuracy plateaued around 63% while validation accuracy collapsed to near 0%, and test accuracy landed at ~49.9% — essentially random guessing. Swapping in a `Bidirectional` wrapper (letting the model read each tweet both forward and backward) fixed this and is why the final model above uses it instead.

## Results Comparison

| Model | Test Accuracy |
|---|---|
| TF-IDF + Logistic Regression (baseline) | ~79% |
| Plain LSTM (unidirectional) | ~50% (failed to converge) |
| **Bidirectional LSTM** (final) | **~77%** |

The simple TF-IDF + Logistic Regression baseline is competitive with the Bidirectional LSTM here — a good reminder to always check a classical baseline before reaching for a deep learning model. The LSTM's main advantage would likely show up with more training data and/or more epochs, or pretrained Arabic embeddings.

## Try It

In [ ]:
sample_tweet = "الخدمة كانت رائعة جدا وممتازة"
print(f"Tweet: {sample_tweet} | Prediction: {predict_sentiment(sample_tweet)}")